# NegMerge — CIFAR-10 Classifier Unlearning

Reproduces the classifier unlearning experiment from the NegMerge paper (ICML 2025).

**Pipeline:**
1. Train a base ResNet18 / VGG16 / Swin-T on full CIFAR-10
2. Fine-tune 30 copies on the **forget set** (10% of training data) with a 10 LR × 3 WD grid
3. Apply **NegMerge** (sign-consensus task vector merging + negate)
4. Apply **Task Arithmetic baseline** (best single-model negation)
5. (Optional) Retrain oracle on retain set only
6. Compare all methods

**Everything is self-contained** — no imports from the repo. Checkpoints are saved to Google Drive.

## 0 · Setup

In [1]:
# Install optional dependency for Swin-T (skip if using resnet18/vgg16)
# !pip install timm -q

from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Mounted at /content/drive
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## 1 · Configuration

Edit this cell to change the model, paths, or hyperparameters.

In [2]:
import os

# -- Model --
MODEL = 'resnet18'          # 'resnet18' | 'vgg16' | 'swin_t'

# -- Paths (Google Drive) --
DRIVE_BASE   = '/content/drive/MyDrive/negmerge_experiment'
DATA_DIR     = '/content/cifar10'               # downloaded locally (faster IO)
CKPT_DIR     = os.path.join(DRIVE_BASE, 'checkpoints', MODEL)
FT_DIR       = os.path.join(CKPT_DIR, 'finetuned')
RESULTS_DIR  = os.path.join(DRIVE_BASE, 'results')

os.makedirs(CKPT_DIR,    exist_ok=True)
os.makedirs(FT_DIR,      exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR,    exist_ok=True)

# -- Training hyperparameters --
TRAIN_EPOCHS    = 200     # base model training epochs
TRAIN_LR        = 0.1
TRAIN_WD        = 5e-4
TRAIN_MOMENTUM  = 0.9

# -- Fine-tuning hyperparameters (27-model grid: 3 x 3 x 3) --
# Fixed settings across all 27 models
FT_LR           = 0.05   # fixed LR for all fine-tuned models

# Axes of diversity (3 x 3 x 3 = 27 combinations)
EPOCHS_GRID     = [40, 50, 60]          # training duration
WD_GRID         = [1e-4, 5e-5, 1e-5]   # weight decay
LS_GRID         = [0.0, 0.05, 0.1]     # label smoothing

# -- Experiment settings --
BATCH_SIZE      = 256     # larger batch is faster on GPU
NUM_WORKERS     = 2
FORGET_FRACTION = 0.1     # 10% of training set is the forget set
SEED            = 42
N_EVAL_POINTS   = 21      # number of alpha values to sweep [0, 1]
RETAIN_THRESHOLD = 0.99   # retain accuracy must be >= this * pretrained_retain
RUN_ORACLE      = True   # set True to also train the retrain oracle (slow)

n_models = len(EPOCHS_GRID) * len(WD_GRID) * len(LS_GRID)
print(f"Model:            {MODEL}")
print(f"Checkpoints dir:  {CKPT_DIR}")
print(f"Fine-tuned dir:   {FT_DIR}")
print(f"Results dir:      {RESULTS_DIR}")
print(f"Fine-tune LR:     {FT_LR}")
print(f"Epochs grid:      {EPOCHS_GRID}")
print(f"WD grid:          {WD_GRID}")
print(f"Label smooth:     {LS_GRID}")
print(f"Total models:     {n_models} ({len(EPOCHS_GRID)} epochs x {len(WD_GRID)} WDs x {len(LS_GRID)} LS)")


Model:            resnet18
Checkpoints dir:  /content/drive/MyDrive/negmerge_experiment/checkpoints/resnet18
Fine-tuned dir:   /content/drive/MyDrive/negmerge_experiment/checkpoints/resnet18/finetuned
Results dir:      /content/drive/MyDrive/negmerge_experiment/results
Fine-tune LR:     0.05
Epochs grid:      [40, 50, 60]
WD grid:          [0.0001, 5e-05, 1e-05]
Label smooth:     [0.0, 0.05, 0.1]
Total models:     27 (3 epochs x 3 WDs x 3 LS)


## 2 · Shared Utilities

All helper functions — dataset loading, model definitions, task vectors, evaluation.

In [3]:
import copy
import gc
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Subset, DataLoader
import torchvision
import torchvision.transforms as transforms
import torchvision.models as tv_models


# ── Constants ───────────────────────────────────────────────────────────────
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)
NUM_CLASSES  = 10


# ── Device ──────────────────────────────────────────────────────────────────
def get_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')

DEVICE = get_device()
print(f"Device: {DEVICE}")


# ── Dataset ──────────────────────────────────────────────────────────────────
def cifar10_transforms():
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
    ])
    eval_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
    ])
    return train_tf, eval_tf


def make_forget_retain_indices(n_total, forget_fraction=0.1, seed=42):
    rng = np.random.default_rng(seed)
    n_forget = int(n_total * forget_fraction)
    forget_idx = rng.choice(n_total, size=n_forget, replace=False)
    retain_idx = np.setdiff1d(np.arange(n_total), forget_idx)
    return forget_idx, retain_idx


def get_cifar10_loaders(data_dir, forget_fraction=0.1, seed=42,
                        batch_size=128, num_workers=4,
                        forget_indices=None, retain_indices=None):
    train_tf, eval_tf = cifar10_transforms()
    full_train_aug  = torchvision.datasets.CIFAR10(data_dir, train=True,  download=True,  transform=train_tf)
    full_train_eval = torchvision.datasets.CIFAR10(data_dir, train=True,  download=False, transform=eval_tf)
    test_dataset    = torchvision.datasets.CIFAR10(data_dir, train=False, download=True,  transform=eval_tf)

    if forget_indices is None or retain_indices is None:
        forget_indices, retain_indices = make_forget_retain_indices(
            len(full_train_aug), forget_fraction, seed
        )

    def loader(ds, shuffle=False):
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                          num_workers=num_workers, pin_memory=True)

    return {
        'full_train':    loader(full_train_aug,                     shuffle=True),
        'forget_train':  loader(Subset(full_train_aug, forget_indices), shuffle=True),
        'retain_train':  loader(Subset(full_train_aug, retain_indices), shuffle=True),
        'forget_eval':   loader(Subset(full_train_eval, forget_indices)),
        'retain_eval':   loader(Subset(full_train_eval, retain_indices)),
        'test':          loader(test_dataset),
        'forget_indices': forget_indices,
        'retain_indices': retain_indices,
        'num_classes':   NUM_CLASSES,
    }


# ── Models ───────────────────────────────────────────────────────────────────
def get_model(model_name, num_classes=10):
    """
    CIFAR-10 adapted architectures (input 32×32).
    ResNet18: 3×3 stem, no maxpool.
    VGG16: adaptive average pool, smaller classifier.
    Swin-T: via timm with img_size=32.
    """
    name = model_name.lower().replace('-', '_')
    if name == 'resnet18':
        model = tv_models.resnet18(weights=None)
        model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        model.maxpool = nn.Identity()
        model.fc      = nn.Linear(model.fc.in_features, num_classes)
    elif name == 'vgg16':
        model = tv_models.vgg16(weights=None)
        model.avgpool    = nn.AdaptiveAvgPool2d((1, 1))
        model.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 512), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )
    elif name in ('swin_t', 'swint'):
        import timm
        model = timm.create_model(
            'swin_tiny_patch4_window7_224', pretrained=False,
            num_classes=num_classes, img_size=32,
        )
    else:
        raise ValueError(f"Unknown model '{model_name}'. Choose: resnet18, vgg16, swin_t")
    return model


# ── Task Vectors ─────────────────────────────────────────────────────────────
class ClassifierTaskVector:
    """
    vector = finetuned_state_dict - pretrained_state_dict.
    Supports negation, scalar multiplication, and apply_to().
    """
    # BatchNorm running stats (running_mean, running_var) are NOT trained by
    # the optimiser -- they track batch statistics during forward passes and
    # drift consistently across all fine-tuned runs.  This means they always
    # pass sign-consensus, but negating them destroys batch-norm inference
    # completely at any alpha > 0.  Exclude them from all task vectors.
    _SKIP_KEYS = ('running_mean', 'running_var', 'num_batches_tracked')

    def __init__(self, pretrained_sd=None, finetuned_sd=None, vector=None):
        if vector is not None:
            self.vector = vector
            return
        with torch.no_grad():
            self.vector = {
                k: finetuned_sd[k].float() - pretrained_sd[k].float()
                for k in pretrained_sd
                if k in finetuned_sd
                and pretrained_sd[k].dtype not in (torch.int64, torch.uint8, torch.bool)
                and not any(s in k for s in ClassifierTaskVector._SKIP_KEYS)
            }

    def __neg__(self):
        return ClassifierTaskVector(vector={k: -v for k, v in self.vector.items()})

    def __mul__(self, scalar):
        return ClassifierTaskVector(vector={k: scalar * v for k, v in self.vector.items()})
    __rmul__ = __mul__

    def apply_to(self, pretrained_sd, scaling_coef=1.0):
        with torch.no_grad():
            return {
                k: (pretrained_sd[k].float() + scaling_coef * self.vector[k]).to(pretrained_sd[k].dtype)
                if k in self.vector else pretrained_sd[k].clone()
                for k in pretrained_sd
            }


def negmerge(task_vectors):
    """
    NegMerge: merge N task vectors using sign-consensus, then negate.
    Only parameters where ALL models agree on direction (same sign) survive.
    Returns the negated merged vector (ready for apply_to).
    """
    n = len(task_vectors)
    merged   = {k: torch.zeros_like(v) for k, v in task_vectors[0].vector.items()}
    sign_sum = {k: torch.zeros_like(v) for k, v in task_vectors[0].vector.items()}

    for tv in task_vectors:
        for k in merged:
            merged[k]   += tv.vector[k]
            sign_sum[k] += torch.sign(tv.vector[k])

    n_total, n_kept = 0, 0
    for k in merged:
        consensus   = torch.abs(sign_sum[k]) == n
        merged[k]   = torch.where(consensus, merged[k] / n, torch.zeros_like(merged[k]))
        n_total    += consensus.numel()
        n_kept     += int(consensus.sum().item())

    sparsity = 1.0 - n_kept / n_total
    print(f"NegMerge sparsity: {100*sparsity:.1f}%  "
          f"({n_kept:,}/{n_total:,} params kept, {n} models merged)")
    return -ClassifierTaskVector(vector=merged)


# ── Evaluation ────────────────────────────────────────────────────────────────
@torch.no_grad()
def accuracy(model, loader, device):
    model.eval().to(device)
    correct, total = 0, 0
    for images, labels in loader:
        preds = model(images.to(device)).argmax(1)
        correct += preds.eq(labels.to(device)).sum().item()
        total   += labels.size(0)
    return correct / total


@torch.no_grad()
def per_sample_losses(model, loader, device):
    model.eval().to(device)
    criterion = nn.CrossEntropyLoss(reduction='none')
    losses = []
    for images, labels in loader:
        losses.extend(criterion(model(images.to(device)), labels.to(device)).cpu().numpy())
    return np.array(losses)


def mia_auc(forget_losses, test_losses):
    """
    Membership Inference Attack AUC. ~0.5 = perfect unlearning, ~1.0 = still memorised.
    Computed without sklearn.
    """
    labels = np.concatenate([np.ones(len(forget_losses)), np.zeros(len(test_losses))])
    scores = np.concatenate([-forget_losses, -test_losses])  # low loss → likely member
    order  = np.argsort(scores)[::-1]
    sl     = labels[order]
    n_pos, n_neg = labels.sum(), len(labels) - labels.sum()
    if n_pos == 0 or n_neg == 0:
        return float('nan')
    tpr = np.cumsum(sl) / n_pos
    fpr = np.cumsum(1 - sl) / n_neg
    trapz_fn = np.trapezoid if hasattr(np, 'trapezoid') else np.trapz
    return float(trapz_fn(tpr, fpr))


def evaluate_unlearning(model, loaders, device, label=''):
    tag = f"[{label}] " if label else ''
    results = {
        'forget_acc': accuracy(model, loaders['forget_eval'], device),
        'retain_acc': accuracy(model, loaders['retain_eval'], device),
        'test_acc':   accuracy(model, loaders['test'],        device),
    }
    fl = per_sample_losses(model, loaders['forget_eval'], device)
    tl = per_sample_losses(model, loaders['test'],        device)
    results['mia_auc'] = mia_auc(fl, tl)
    print(f"\n{tag}Forget acc:  {results['forget_acc']:.4f}  (chance={1/NUM_CLASSES:.4f})")
    print(f"{tag}Retain acc:  {results['retain_acc']:.4f}")
    print(f"{tag}Test acc:    {results['test_acc']:.4f}")
    print(f"{tag}MIA AUC:     {results['mia_auc']:.4f}  (0.5=unlearned, 1.0=memorised)")
    return results


def clear_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Utilities loaded.")

Device: cuda
Utilities loaded.


## 3 · Load Data & Preload to GPU

Creates the deterministic forget/retain split (saved to Drive for reproducibility),
then preloads the **entire CIFAR-10 dataset into GPU memory** (~720 MB).

With all data on the GPU, each training batch is just a tensor slice with zero
CPU-to-GPU transfer overhead. This raises GPU utilisation from ~1% to ~80–90%
and reduces each epoch from ~60 s to **under 2 s**.


In [4]:
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Load split indices ─────────────────────────────────────────────────────
forget_idx_path = os.path.join(CKPT_DIR, "forget_indices.npy")
retain_idx_path = os.path.join(CKPT_DIR, "retain_indices.npy")

if os.path.exists(forget_idx_path) and os.path.exists(retain_idx_path):
    forget_idx = np.load(forget_idx_path)
    retain_idx = np.load(retain_idx_path)
    print("Loaded existing split from Drive.")
else:
    forget_idx, retain_idx = make_forget_retain_indices(
        n_total=50000, forget_fraction=FORGET_FRACTION, seed=SEED
    )
    np.save(forget_idx_path, forget_idx)
    np.save(retain_idx_path, retain_idx)
    print("Created new split and saved to Drive.")

print(f"Forget: {len(forget_idx):,}  Retain: {len(retain_idx):,}")

# ── Preload entire CIFAR-10 into GPU memory ─────────────────────────────────
# CIFAR-10 is only ~720 MB total — fits easily in T4 16 GB.
# With data already on GPU, each batch is just a tensor slice: zero transfer
# overhead per batch, GPU utilisation goes from <1% to ~80-90%.
print("Preloading CIFAR-10 into GPU memory (one-time, ~30 sec)...")

train_tf, eval_tf = cifar10_transforms()
raw_train_aug  = torchvision.datasets.CIFAR10(DATA_DIR, train=True,  download=True,  transform=train_tf)
raw_train_eval = torchvision.datasets.CIFAR10(DATA_DIR, train=True,  download=False, transform=eval_tf)
raw_test       = torchvision.datasets.CIFAR10(DATA_DIR, train=False, download=True,  transform=eval_tf)

def dataset_to_gpu(dataset, desc=""):
    """Load a torchvision dataset into two GPU tensors (images, labels)."""
    loader = DataLoader(dataset, batch_size=2000, num_workers=2, pin_memory=True)
    imgs, lbls = [], []
    for x, y in loader:
        imgs.append(x.to(DEVICE))
        lbls.append(y.to(DEVICE))
    imgs = torch.cat(imgs)
    lbls = torch.cat(lbls)
    if desc:
        print(f"  {desc}: {tuple(imgs.shape)}  ({imgs.nbytes/1e6:.0f} MB)")
    return imgs, lbls

train_aug_imgs,  train_aug_lbls  = dataset_to_gpu(raw_train_aug,  "train (augmented)")
train_eval_imgs, train_eval_lbls = dataset_to_gpu(raw_train_eval, "train (eval)")
test_imgs,       test_lbls       = dataset_to_gpu(raw_test,        "test")
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ── Build index tensors for forget / retain subsets ─────────────────────────
forget_idx_t = torch.from_numpy(forget_idx).long().to(DEVICE)
retain_idx_t = torch.from_numpy(retain_idx).long().to(DEVICE)

# ── In-memory DataLoaders (num_workers=0 — data is already on GPU) ──────────
class GPUTensorDataset(torch.utils.data.Dataset):
    def __init__(self, imgs, lbls):
        self.imgs = imgs
        self.lbls = lbls
    def __len__(self):
        return len(self.lbls)
    def __getitem__(self, i):
        return self.imgs[i], self.lbls[i]

def gpu_loader(imgs, lbls, shuffle=False):
    ds = GPUTensorDataset(imgs, lbls)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=0, pin_memory=False)

# Training loaders use the CPU dataset with live RandomCrop/Flip each epoch.
# Preloading the augmented split once (as done above for train_aug_imgs) fixes
# the random transform at load time, killing augmentation diversity.
train_tf, _ = cifar10_transforms()
cpu_train_aug = torchvision.datasets.CIFAR10(DATA_DIR, train=True, download=False, transform=train_tf)

def cpu_loader(dataset, shuffle=False):
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True)

loaders = {
    # Training: CPU DataLoaders so augmentation is re-sampled every epoch
    "full_train":    cpu_loader(cpu_train_aug, shuffle=True),
    "forget_train":  cpu_loader(Subset(cpu_train_aug, forget_idx), shuffle=True),
    "retain_train":  cpu_loader(Subset(cpu_train_aug, retain_idx), shuffle=True),
    # Evaluation: GPU tensors (eval_tf has no randomness, safe to preload)
    "forget_eval":   gpu_loader(train_eval_imgs[forget_idx_t], train_eval_lbls[forget_idx_t]),
    "retain_eval":   gpu_loader(train_eval_imgs[retain_idx_t], train_eval_lbls[retain_idx_t]),
    "test":          gpu_loader(test_imgs, test_lbls),
    "forget_indices": forget_idx,
    "retain_indices": retain_idx,
    "num_classes":   NUM_CLASSES,
}
print("Training loaders: CPU (live augmentation each epoch)")
print("Eval loaders:     GPU tensors (no augmentation, fast inference)")


Loaded existing split from Drive.
Forget: 5,000  Retain: 45,000
Preloading CIFAR-10 into GPU memory (one-time, ~30 sec)...


100%|██████████| 170M/170M [00:06<00:00, 28.2MB/s]


  train (augmented): (50000, 3, 32, 32)  (614 MB)
  train (eval): (50000, 3, 32, 32)  (614 MB)
  test: (10000, 3, 32, 32)  (123 MB)
GPU memory allocated: 1.35 GB
Training loaders: CPU (live augmentation each epoch)
Eval loaders:     GPU tensors (no augmentation, fast inference)


## 4 · Step 1 — Train Base Model

Trains the model on the **full** CIFAR-10 training set for 200 epochs.
This is the starting point for all unlearning methods.

**Skipped automatically if `pretrained_best.pt` already exists on Drive.**

Expected accuracy targets (training from scratch, no pretrained weights):
- ResNet18: ~94%
- VGG16: ~93%
- Swin-T: ~88%

In [5]:
pretrained_path = os.path.join(CKPT_DIR, 'pretrained_best.pt')

if os.path.exists(pretrained_path):
    print(f"Found pretrained_best.pt on Drive — skipping training.")
    pretrained_sd = torch.load(pretrained_path, map_location='cpu', weights_only=True)
else:
    print(f"Training {MODEL} on full CIFAR-10 for {TRAIN_EPOCHS} epochs...")
    print(f"LR={TRAIN_LR}  WD={TRAIN_WD}  Batch={BATCH_SIZE}\n")

    model     = get_model(MODEL, num_classes=NUM_CLASSES).to(DEVICE)
    n_params  = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Parameters: {n_params:,}")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=TRAIN_LR,
                          momentum=TRAIN_MOMENTUM, weight_decay=TRAIN_WD)
    scheduler = CosineAnnealingLR(optimizer, T_max=TRAIN_EPOCHS, eta_min=1e-4)

    best_acc, best_epoch = 0.0, 0
    history = []


    for epoch in range(1, TRAIN_EPOCHS + 1):
        # ── Train one epoch ──────────────────────────────────────────────────
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for images, labels in loaders['full_train']:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            out  = model(images)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct    += out.argmax(1).eq(labels).sum().item()
            total      += labels.size(0)
        scheduler.step()
        print(f"Epoch {epoch:3d}/{TRAIN_EPOCHS}")
        # ── Evaluate every 20 epochs ─────────────────────────────────────────
        if epoch % 20 == 0 or epoch == TRAIN_EPOCHS:
            test_acc   = accuracy(model, loaders['test'],        DEVICE)
            forget_acc = accuracy(model, loaders['forget_eval'], DEVICE)
            retain_acc = accuracy(model, loaders['retain_eval'], DEVICE)
            train_acc  = correct / total
            lr_now     = scheduler.get_last_lr()[0]

            history.append({
                'epoch': epoch, 'train_acc': train_acc, 'test_acc': test_acc,
                'forget_acc': forget_acc, 'retain_acc': retain_acc,
                'train_loss': total_loss / total, 'lr': lr_now,
            })
            print(f"Epoch {epoch:3d}/{TRAIN_EPOCHS}  "
                  f"loss={total_loss/total:.4f}  train={train_acc:.4f}  "
                  f"test={test_acc:.4f}  forget={forget_acc:.4f}  "
                  f"retain={retain_acc:.4f}  lr={lr_now:.2e}")

            if test_acc > best_acc:
                best_acc   = test_acc
                best_epoch = epoch
                torch.save(model.state_dict(), pretrained_path)

    # Save final checkpoint and metadata
    torch.save(model.state_dict(), os.path.join(CKPT_DIR, 'pretrained_final.pt'))
    with open(os.path.join(CKPT_DIR, 'train_history.json'), 'w') as f:
        json.dump(history, f, indent=2)

    print(f"\nBest test accuracy: {best_acc:.4f} at epoch {best_epoch}")
    print(f"Saved to {pretrained_path}")

    pretrained_sd = torch.load(pretrained_path, map_location='cpu', weights_only=True)
    del model
    clear_cache()

# Verify pretrained model
check_model = get_model(MODEL, num_classes=NUM_CLASSES)
check_model.load_state_dict(pretrained_sd)
pretrained_retain = accuracy(check_model, loaders['retain_eval'], DEVICE)
pretrained_forget = accuracy(check_model, loaders['forget_eval'], DEVICE)
pretrained_test   = accuracy(check_model, loaders['test'],        DEVICE)
del check_model
clear_cache()

print(f"\nPretrained model:")
print(f"  Test acc:   {pretrained_test:.4f}")
print(f"  Forget acc: {pretrained_forget:.4f}  ← should be high (model knows these)")
print(f"  Retain acc: {pretrained_retain:.4f}")

Found pretrained_best.pt on Drive — skipping training.

Pretrained model:
  Test acc:   0.9520
  Forget acc: 1.0000  ← should be high (model knows these)
  Retain acc: 1.0000


## 5 · Step 2 — Fine-tune 30 Models on the Forget Set

Each model starts from `pretrained_best.pt` and is fine-tuned on the **forget set only**
using AdamW with a different (LR, WD) hyperparameter combination.

This generates the diverse task vectors that NegMerge merges.

**Already-completed checkpoints are skipped** — safe to interrupt and resume.

In [6]:
import itertools

hp_grid = list(itertools.product(EPOCHS_GRID, WD_GRID, LS_GRID))
print(f"Fine-tuning {len(hp_grid)} models (fixed LR={FT_LR}, optimizer=AdamW, no LR decay)")
print(f"Epochs grid:    {EPOCHS_GRID}")
print(f"WD grid:        {WD_GRID}")
print(f"Label smooth:   {LS_GRID}\n")

finetune_summary = []

for i, (epochs, wd, ls) in enumerate(hp_grid):
    tag       = f"ep{epochs}_wd{wd:.0e}_ls{ls:.2f}".replace('e-0', 'e-').replace('e+0', 'e')
    ckpt_path = os.path.join(FT_DIR, f'{tag}.pt')

    if os.path.exists(ckpt_path):
        print(f"[{i+1:02d}/{len(hp_grid)}] {tag} -- already exists, skipping")
        finetune_summary.append({'tag': tag, 'epochs': epochs, 'wd': wd, 'ls': ls, 'skipped': True})
        continue

    print(f"[{i+1:02d}/{len(hp_grid)}] {tag}  epochs={epochs}  wd={wd:.0e}  ls={ls:.2f}", end='', flush=True)

    # Build a fresh copy from the pretrained weights
    model = get_model(MODEL, num_classes=NUM_CLASSES)
    model.load_state_dict(copy.deepcopy(pretrained_sd))
    model = model.to(DEVICE)
    model.train()

    criterion = nn.CrossEntropyLoss(label_smoothing=ls)
    optimizer = optim.AdamW(model.parameters(), lr=FT_LR,
                           weight_decay=wd)

    final_loss, final_acc = 0.0, 0.0
    for epoch in range(1, epochs + 1):
        total_loss, correct, total = 0.0, 0, 0
        for images, labels in loaders['forget_train']:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            out  = model(images)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct    += out.argmax(1).eq(labels).sum().item()
            total      += labels.size(0)
        if epoch == epochs:
            final_loss = total_loss / total
            final_acc  = correct / total

    torch.save(model.state_dict(), ckpt_path)
    finetune_summary.append({
        'tag': tag, 'epochs': epochs, 'wd': wd, 'ls': ls,
        'forget_train_loss': final_loss, 'forget_train_acc': final_acc,
    })
    print(f"  -> loss={final_loss:.4f}  acc={final_acc:.4f}")

    del model
    clear_cache()

with open(os.path.join(FT_DIR, 'finetune_summary.json'), 'w') as f:
    json.dump(finetune_summary, f, indent=2)

ckpt_paths = sorted([
    os.path.join(FT_DIR, f) for f in os.listdir(FT_DIR) if f.endswith('.pt')
])
print(f'\nDone. {len(ckpt_paths)} finetuned checkpoints available.')


Fine-tuning 27 models (fixed LR=0.05, momentum=0.9)
Epochs grid:    [40, 50, 60]
WD grid:        [0.0001, 5e-05, 1e-05]
Label smooth:   [0.0, 0.05, 0.1]

[01/27] ep40_wd1e-4_ls0.00 -- already exists, skipping
[02/27] ep40_wd1e-4_ls0.05 -- already exists, skipping
[03/27] ep40_wd1e-4_ls0.10 -- already exists, skipping
[04/27] ep40_wd5e-5_ls0.00 -- already exists, skipping
[05/27] ep40_wd5e-5_ls0.05 -- already exists, skipping
[06/27] ep40_wd5e-5_ls0.10 -- already exists, skipping
[07/27] ep40_wd1e-5_ls0.00 -- already exists, skipping
[08/27] ep40_wd1e-5_ls0.05 -- already exists, skipping
[09/27] ep40_wd1e-5_ls0.10 -- already exists, skipping
[10/27] ep50_wd1e-4_ls0.00 -- already exists, skipping
[11/27] ep50_wd1e-4_ls0.05 -- already exists, skipping
[12/27] ep50_wd1e-4_ls0.10 -- already exists, skipping
[13/27] ep50_wd5e-5_ls0.00 -- already exists, skipping
[14/27] ep50_wd5e-5_ls0.05 -- already exists, skipping
[15/27] ep50_wd5e-5_ls0.10 -- already exists, skipping
[16/27] ep50_wd1e-5_l

## 6 · Step 3 — NegMerge

Builds task vectors from all 30 fine-tuned models, applies sign-consensus masking,
averages the surviving parameters, negates the result, and finds the optimal scaling
coefficient α via a sweep on the forget/retain sets.

In [7]:
print("=== Step 3: NegMerge ===")
print(f"Loading {len(ckpt_paths)} finetuned checkpoints...")

# Build task vectors
task_vectors = []
for ckpt_path in ckpt_paths:
    ft_sd = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    task_vectors.append(ClassifierTaskVector(pretrained_sd=pretrained_sd, finetuned_sd=ft_sd))
    del ft_sd
print(f"Built {len(task_vectors)} task vectors.")

# NegMerge: sign-consensus + negate
print("\nApplying sign-consensus merge...")
negmerge_tv = negmerge(task_vectors)

# -- Sparsity guard --------------------------------------------------------
# Healthy sparsity with the fixed-LR grid should be 30-70%.
# >95% means the task vectors still point in opposite directions.
tv_keys = list(negmerge_tv.vector.keys())
n_nonzero = sum(int((negmerge_tv.vector[k] != 0).sum().item()) for k in tv_keys)
n_total_params = sum(negmerge_tv.vector[k].numel() for k in tv_keys)
sparsity_pct = 100.0 * (1 - n_nonzero / n_total_params)
print(f"Merged vector: {n_nonzero:,} / {n_total_params:,} non-zero params  "
      f"(sparsity {sparsity_pct:.1f}%)")
if sparsity_pct > 99.0:
    raise RuntimeError(
        f"NegMerge sparsity is {sparsity_pct:.1f}% -- almost no parameters "
        "survived sign consensus.  The task vectors are pointing in opposite "
        "directions.  Check the fine-tuning setup."
    )

# -- Coefficient sweep ------------------------------------------------------
# Stage 1: broad sweep over [0, 1] to locate where retain accuracy drops.
# Stage 2: fine-grained sweep below that cliff to find the best alpha.
# Coefficient sweep: find α that minimises forget acc while keeping retain acc high
print(f"\nStage 1 — broad sweep (α ∈ [0,1], {N_EVAL_POINTS} points)...")
retain_threshold_abs = RETAIN_THRESHOLD * pretrained_retain
print(f"Retain threshold: {retain_threshold_abs:.4f} ({RETAIN_THRESHOLD:.0%} × {pretrained_retain:.4f})")

def eval_alpha(alpha):
    sd    = negmerge_tv.apply_to(pretrained_sd, scaling_coef=float(alpha))
    model = get_model(MODEL, num_classes=NUM_CLASSES)
    model.load_state_dict(sd)
    del sd
    model = model.to(DEVICE)
    fa = accuracy(model, loaders['forget_eval'], DEVICE)
    ra = accuracy(model, loaders['retain_eval'], DEVICE)
    del model
    clear_cache()
    return fa, ra

nm_sweep = []
cliff_alpha = 1.0
for alpha in np.linspace(0.0, 1.0, N_EVAL_POINTS):
    fa, ra = eval_alpha(alpha)
    nm_sweep.append({'alpha': float(alpha), 'forget_acc': fa, 'retain_acc': ra})
    print(f"  α={alpha:.2f}  forget={fa:.4f}  retain={ra:.4f}")
    if ra < retain_threshold_abs and cliff_alpha == 1.0:
        cliff_alpha = float(alpha)

# Stage 2: fine-grained sweep below the cliff
if 0.0 < cliff_alpha < 1.0:
    print(f"\nStage 2 — fine sweep (α ∈ [0, {cliff_alpha:.2f}], 21 points)...")
    for alpha in np.linspace(0.0, cliff_alpha, 21)[1:]:
        fa, ra = eval_alpha(alpha)
        nm_sweep.append({'alpha': float(alpha), 'forget_acc': fa, 'retain_acc': ra})
        print(f"  α={alpha:.3f}  forget={fa:.4f}  retain={ra:.4f}")

# Pick best alpha
valid = [s for s in nm_sweep if s['retain_acc'] >= retain_threshold_abs]
if not valid:
    raise RuntimeError(
        "No alpha value met the retain threshold.  "
        "The negated task vector is too destructive at all scales.  "
        "Try lowering RETAIN_THRESHOLD or checking the fine-tuning setup."
    )
nm_best = min(valid, key=lambda s: s['forget_acc'])

print(f"\nBest α: {nm_best['alpha']:.4f}  "
      f"forget={nm_best['forget_acc']:.4f}  retain={nm_best['retain_acc']:.4f}")

# Final evaluation at best alpha
print("\n=== NegMerge Final Evaluation ===")
best_sd = negmerge_tv.apply_to(pretrained_sd, scaling_coef=nm_best['alpha'])
nm_model = get_model(MODEL, num_classes=NUM_CLASSES)
nm_model.load_state_dict(best_sd)
nm_results = evaluate_unlearning(nm_model, loaders, DEVICE, label='NegMerge')

# Save unlearned model
nm_save_path = os.path.join(RESULTS_DIR, f'{MODEL}_negmerge_unlearned.pt')
torch.save(best_sd, nm_save_path)

# Pretrained reference
print("\n=== Pretrained Reference ===")
pretrained_model = get_model(MODEL, num_classes=NUM_CLASSES)
pretrained_model.load_state_dict(pretrained_sd)
pretrained_results = evaluate_unlearning(pretrained_model, loaders, DEVICE, label='Pretrained')
del pretrained_model, nm_model
clear_cache()

# Save NegMerge results
nm_out = {
    'method':           'NegMerge',
    'model':            MODEL,
    'n_task_vectors':   len(task_vectors),
    'best_alpha':       nm_best['alpha'],
    'retain_threshold': RETAIN_THRESHOLD,
    'pretrained':       pretrained_results,
    'unlearned':        nm_results,
    'coefficient_sweep': nm_sweep,
}
with open(os.path.join(RESULTS_DIR, f'{MODEL}_negmerge_results.json'), 'w') as f:
    json.dump(nm_out, f, indent=2)
print(f"\nResults saved to {RESULTS_DIR}/{MODEL}_negmerge_results.json")

=== Step 3: NegMerge ===
Loading 27 finetuned checkpoints...
Built 27 task vectors.

Applying sign-consensus merge...
NegMerge sparsity: 98.5%  (170,945/11,173,962 params kept, 27 models merged)
Merged vector: 170,945 / 11,173,962 non-zero params  (sparsity 98.5%)

Stage 1 — broad sweep (α ∈ [0,1], 21 points)...
Retain threshold: 0.9500 (95% × 1.0000)
  α=0.00  forget=1.0000  retain=1.0000
  α=0.05  forget=1.0000  retain=1.0000
  α=0.10  forget=1.0000  retain=1.0000
  α=0.15  forget=0.9998  retain=1.0000
  α=0.20  forget=0.9992  retain=0.9995
  α=0.25  forget=0.9946  retain=0.9976
  α=0.30  forget=0.9896  retain=0.9922
  α=0.35  forget=0.9772  retain=0.9803
  α=0.40  forget=0.9538  retain=0.9602
  α=0.45  forget=0.9208  retain=0.9285
  α=0.50  forget=0.8732  retain=0.8845
  α=0.55  forget=0.8224  retain=0.8276
  α=0.60  forget=0.7542  retain=0.7604
  α=0.65  forget=0.6810  retain=0.6869
  α=0.70  forget=0.6038  retain=0.6070
  α=0.75  forget=0.5350  retain=0.5333
  α=0.80  forget=0.459

## 7 · Step 4 — Task Arithmetic Baseline

Evaluates each of the 30 checkpoints individually as a Task Arithmetic candidate.
For each checkpoint, negates its task vector and sweeps α.
Picks the single (checkpoint, α) pair that best minimises forget accuracy.

In [8]:
print("=== Step 4: Task Arithmetic Baseline ===")
alphas = np.linspace(0.0, 1.0, N_EVAL_POINTS)
retain_threshold_abs = RETAIN_THRESHOLD * pretrained_retain

best_result  = None   # {'ckpt', 'tag', 'alpha', 'forget_acc', 'retain_acc'}
all_per_ckpt = {}

for ckpt_path in ckpt_paths:
    tag   = os.path.basename(ckpt_path).replace('.pt', '')
    ft_sd = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    neg_tv = -ClassifierTaskVector(pretrained_sd=pretrained_sd, finetuned_sd=ft_sd)
    del ft_sd
    print(f"  {tag}", end='', flush=True)

    best_local = None
    for alpha in alphas:
        sd    = neg_tv.apply_to(pretrained_sd, scaling_coef=float(alpha))
        model = get_model(MODEL, num_classes=NUM_CLASSES)
        model.load_state_dict(sd)
        del sd
        model = model.to(DEVICE)
        fa = accuracy(model, loaders['forget_eval'], DEVICE)
        ra = accuracy(model, loaders['retain_eval'], DEVICE)
        del model
        clear_cache()

        if ra >= retain_threshold_abs:
            if best_local is None or fa < best_local['forget_acc']:
                best_local = {'alpha': float(alpha), 'forget_acc': fa, 'retain_acc': ra}

    if best_local is None:
        print(f"  → no valid α")
        all_per_ckpt[tag] = None
        continue

    print(f"  → best α={best_local['alpha']:.2f}  forget={best_local['forget_acc']:.4f}  "
          f"retain={best_local['retain_acc']:.4f}")
    all_per_ckpt[tag] = best_local

    if best_result is None or best_local['forget_acc'] < best_result['forget_acc']:
        best_result = {**best_local, 'ckpt': ckpt_path, 'tag': tag}

if best_result is None:
    print("ERROR: No valid (checkpoint, alpha) found. Try lower --retain-threshold.")
else:
    print(f"\nBest checkpoint: {best_result['tag']}")
    print(f"Best α={best_result['alpha']:.4f}  "
          f"forget={best_result['forget_acc']:.4f}  retain={best_result['retain_acc']:.4f}")

    # Final evaluation
    print("\n=== Task Arithmetic Final Evaluation ===")
    best_ft_sd = torch.load(best_result['ckpt'], map_location='cpu', weights_only=True)
    best_neg_tv = -ClassifierTaskVector(pretrained_sd=pretrained_sd, finetuned_sd=best_ft_sd)
    final_sd = best_neg_tv.apply_to(pretrained_sd, scaling_coef=best_result['alpha'])
    ta_model = get_model(MODEL, num_classes=NUM_CLASSES)
    ta_model.load_state_dict(final_sd)
    ta_results = evaluate_unlearning(ta_model, loaders, DEVICE, label='TaskArithmetic')
    del ta_model
    clear_cache()

    ta_save_path = os.path.join(RESULTS_DIR, f'{MODEL}_baseline_unlearned.pt')
    torch.save(final_sd, ta_save_path)

    ta_out = {
        'method':          'TaskArithmetic',
        'model':           MODEL,
        'best_checkpoint': best_result['tag'],
        'best_alpha':      best_result['alpha'],
        'retain_threshold': RETAIN_THRESHOLD,
        'pretrained':      pretrained_results,
        'unlearned':       ta_results,
        'per_checkpoint_results': {k: v for k, v in all_per_ckpt.items() if v},
    }
    with open(os.path.join(RESULTS_DIR, f'{MODEL}_baseline_results.json'), 'w') as f:
        json.dump(ta_out, f, indent=2)
    print(f"\nResults saved to {RESULTS_DIR}/{MODEL}_baseline_results.json")

=== Step 4: Task Arithmetic Baseline ===
  ep40_wd1e-4_ls0.00  → best α=0.10  forget=0.9644  retain=0.9751
  ep40_wd1e-4_ls0.05  → best α=0.20  forget=0.9540  retain=0.9721
  ep40_wd1e-4_ls0.10  → best α=0.30  forget=0.9318  retain=0.9644
  ep40_wd1e-5_ls0.00  → best α=0.10  forget=0.9590  retain=0.9728
  ep40_wd1e-5_ls0.05  → best α=0.15  forget=0.9790  retain=0.9891
  ep40_wd1e-5_ls0.10  → best α=0.30  forget=0.9234  retain=0.9556
  ep40_wd5e-5_ls0.00  → best α=0.10  forget=0.9624  retain=0.9730
  ep40_wd5e-5_ls0.05  → best α=0.20  forget=0.9452  retain=0.9638
  ep40_wd5e-5_ls0.10  → best α=0.25  forget=0.9582  retain=0.9813
  ep50_wd1e-4_ls0.00  → best α=0.10  forget=0.9714  retain=0.9811
  ep50_wd1e-4_ls0.05  → best α=0.20  forget=0.9294  retain=0.9569
  ep50_wd1e-4_ls0.10  → best α=0.30  forget=0.9268  retain=0.9579
  ep50_wd1e-5_ls0.00  → best α=0.05  forget=0.9976  retain=0.9993
  ep50_wd1e-5_ls0.05  → best α=0.15  forget=0.9764  retain=0.9868
  ep50_wd1e-5_ls0.10  → best α=0.25

## 8 · Step 5 (Optional) — Retrain Oracle

Trains a model from scratch on the **retain set only** — never exposed to the forget set.
This is the gold standard: a model that genuinely has no knowledge of the forget data.

NegMerge and Task Arithmetic are considered good unlearning methods if they approach
this oracle.

**Set `RUN_ORACLE = True` in the configuration cell to enable this step.**
It takes roughly as long as training the base model (~same epochs, smaller dataset).

In [9]:
if not RUN_ORACLE:
    print("Skipping oracle (set RUN_ORACLE = True in the config cell to enable).")
else:
    print(f"=== Step 5: Retrain Oracle (retain set only) ===")
    print(f"Training {MODEL} on {len(retain_idx):,} retain samples for {TRAIN_EPOCHS} epochs...")

    oracle_path = os.path.join(CKPT_DIR, 'oracle_retrained.pt')

    if os.path.exists(oracle_path):
        print("Found oracle_retrained.pt on Drive — skipping training.")
        oracle_sd = torch.load(oracle_path, map_location='cpu', weights_only=True)
    else:
        oracle_model = get_model(MODEL, num_classes=NUM_CLASSES).to(DEVICE)
        criterion    = nn.CrossEntropyLoss()
        optimizer    = optim.SGD(oracle_model.parameters(), lr=TRAIN_LR,
                                 momentum=TRAIN_MOMENTUM, weight_decay=TRAIN_WD)
        scheduler    = CosineAnnealingLR(optimizer, T_max=TRAIN_EPOCHS, eta_min=1e-4)

        best_acc, best_sd = 0.0, None
        for epoch in range(1, TRAIN_EPOCHS + 1):
            oracle_model.train()
            for images, labels in loaders['retain_train']:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()
                loss = criterion(oracle_model(images), labels)
                loss.backward()
                optimizer.step()
            scheduler.step()

            if epoch % 20 == 0 or epoch == TRAIN_EPOCHS:
                test_acc = accuracy(oracle_model, loaders['test'], DEVICE)
                print(f"Epoch {epoch:3d}/{TRAIN_EPOCHS}  test_acc={test_acc:.4f}")
                if test_acc > best_acc:
                    best_acc = test_acc
                    best_sd  = {k: v.clone() for k, v in oracle_model.state_dict().items()}

        oracle_model.load_state_dict(best_sd)
        torch.save(best_sd, oracle_path)
        oracle_sd = best_sd
        del oracle_model
        clear_cache()

    # Evaluate oracle
    print("\n=== Oracle Evaluation ===")
    oracle_model = get_model(MODEL, num_classes=NUM_CLASSES)
    oracle_model.load_state_dict(oracle_sd)
    oracle_results = evaluate_unlearning(oracle_model, loaders, DEVICE, label='Oracle')
    del oracle_model
    clear_cache()

    oracle_out = {'method': 'RetrainOracle', 'model': MODEL, **oracle_results}
    with open(os.path.join(RESULTS_DIR, f'{MODEL}_oracle_results.json'), 'w') as f:
        json.dump(oracle_out, f, indent=2)
    print(f"Oracle saved to {RESULTS_DIR}/{MODEL}_oracle_results.json")

=== Step 5: Retrain Oracle (retain set only) ===
Training resnet18 on 45,000 retain samples for 200 epochs...
Epoch  20/200  test_acc=0.8271
Epoch  40/200  test_acc=0.8737
Epoch  60/200  test_acc=0.8673
Epoch  80/200  test_acc=0.8679
Epoch 100/200  test_acc=0.8860
Epoch 120/200  test_acc=0.9129
Epoch 140/200  test_acc=0.9243
Epoch 160/200  test_acc=0.9438
Epoch 180/200  test_acc=0.9493
Epoch 200/200  test_acc=0.9495

=== Oracle Evaluation ===

[Oracle] Forget acc:  0.9512  (chance=0.1000)
[Oracle] Retain acc:  1.0000
[Oracle] Test acc:    0.9495
[Oracle] MIA AUC:     0.4950  (0.5=unlearned, 1.0=memorised)
Oracle saved to /content/drive/MyDrive/negmerge_experiment/results/resnet18_oracle_results.json


## 9 · Step 6 — Comparison Table

In [10]:
def load_json(path):
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)

def fmt(val, pct=True):
    if val is None: return '   N/A  '
    return f'{val*100:7.2f}%' if pct else f'{val:7.4f} '

d = RESULTS_DIR
negmerge_data = load_json(os.path.join(d, f'{MODEL}_negmerge_results.json'))
baseline_data = load_json(os.path.join(d, f'{MODEL}_baseline_results.json'))
oracle_data   = load_json(os.path.join(d, f'{MODEL}_oracle_results.json'))

rows = []
if negmerge_data:  rows.append(('NegMerge',      negmerge_data.get('unlearned', {})))
if baseline_data:  rows.append(('TaskArithmetic', baseline_data.get('unlearned', {})))
if oracle_data:    rows.append(('RetrainOracle',  oracle_data))

print(f"\n{'='*75}")
print(f"  CIFAR-10 Unlearning Results — {MODEL}")
print(f"{'='*75}")
print(f"  {'Method':20s}  {'Forget↓':>9}  {'Retain↑':>9}  {'Test↑':>9}  {'MIA AUC~0.5':>12}")
print(f"  {'-'*20}  {'-'*9}  {'-'*9}  {'-'*9}  {'-'*12}")
for method, res in rows:
    print(f"  {method:20s}  {fmt(res.get('forget_acc'))}  "
          f"{fmt(res.get('retain_acc'))}  {fmt(res.get('test_acc'))}  "
          f"{fmt(res.get('mia_auc'), pct=False)}")
print(f"  {'Random chance':20s}  {fmt(1/NUM_CLASSES)}")
print(f"{'='*75}")

# Pretrained reference
if negmerge_data and negmerge_data.get('pretrained'):
    r = negmerge_data['pretrained']
    print(f"\n  Pretrained (reference, no unlearning):")
    print(f"    Forget={fmt(r.get('forget_acc')).strip()}  "
          f"Retain={fmt(r.get('retain_acc')).strip()}  "
          f"Test={fmt(r.get('test_acc')).strip()}  "
          f"MIA={fmt(r.get('mia_auc'), pct=False).strip()}")

# Save combined
combined = {method: {'unlearned': res} for method, res in rows}
out_path = os.path.join(d, f'{MODEL}_comparison.json')
with open(out_path, 'w') as f:
    json.dump(combined, f, indent=2)
print(f"\nCombined results saved to {out_path}")


  CIFAR-10 Unlearning Results — resnet18
  Method                  Forget↓    Retain↑      Test↑   MIA AUC~0.5
  --------------------  ---------  ---------  ---------  ------------
  NegMerge                95.10%    95.78%    88.66%   0.5392 
  TaskArithmetic          91.82%    95.34%    88.15%   0.5200 
  RetrainOracle           95.12%   100.00%    94.95%   0.4950 
  Random chance           10.00%

  Pretrained (reference, no unlearning):
    Forget=100.00%  Retain=100.00%  Test=95.20%  MIA=0.5948

Combined results saved to /content/drive/MyDrive/negmerge_experiment/results/resnet18_comparison.json


## 10 · Download Checkpoints

Zip the finetuned checkpoints and download them so you can run NegMerge locally
without re-training.

In [11]:
from google.colab import files
import zipfile

# ── Option A: Download all finetuned checkpoints + pretrained ───────────────
zip_path = f'/content/{MODEL}_checkpoints.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Pretrained base model
    for fname in ['pretrained_best.pt', 'forget_indices.npy', 'retain_indices.npy']:
        p = os.path.join(CKPT_DIR, fname)
        if os.path.exists(p):
            zf.write(p, fname)
    # All finetuned checkpoints
    for fname in os.listdir(FT_DIR):
        if fname.endswith('.pt'):
            zf.write(os.path.join(FT_DIR, fname), f'finetuned/{fname}')
    # Results
    for fname in os.listdir(RESULTS_DIR):
        zf.write(os.path.join(RESULTS_DIR, fname), f'results/{fname}')

print(f"Created {zip_path}")
files.download(zip_path)

Created /content/resnet18_checkpoints.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>